In [ ]:
import cv2
import numpy as np
from ultralytics import SAM

# 全局变量存储点和标签
points = []
labels = []

def mouse_callback(event, x, y, flags, param):
    """鼠标事件回调函数"""
    global points, labels
    
    if event == cv2.EVENT_LBUTTONDOWN:  # 左键点击 - 前景点
        points.append([x, y])
        labels.append(1)
        print(f"添加前景点: ({x}, {y}), 总共{len(points)}个点")
    elif event == cv2.EVENT_RBUTTONDOWN:  # 右键点击 - 背景点
        points.append([x, y])
        labels.append(0)
        print(f"添加背景点: ({x}, {y}), 总共{len(points)}个点")
    elif event == cv2.EVENT_MBUTTONDOWN:  # 中键点击 - 清除
        points.clear()
        labels.clear()
        print("清除所有点")

def main():
    # 加载SAM模型
    sam_model = SAM('/home/weiyn/projects/ai-annotation-studio-service/models/sam_models/sam3.pt')  # 根据需要选择合适的模型
    
    # 加载图像
    img_path = "/home/weiyn/projects/ai-annotation-studio-service/test/images/ScreenShot_2025-12-26_151030_700.png"
    image = cv2.imread(img_path)
    if image is None:
        print(f"无法加载图像: {img_path}")
        return
    
    # 设置鼠标回调
    cv2.namedWindow('SAM3 Interactive')
    cv2.setMouseCallback('SAM3 Interactive', mouse_callback)
    
    print("交互说明:")
    print("- 左键点击选择目标区域")
    print("- 右键点击排除背景区域")
    print("- 按's'键执行分割")
    print("- 按'q'键退出")
    
    while True:
        # 复制图像以绘制点
        disp_img = image.copy()
        
        # 绘制已选择的点
        for point, label in zip(points, labels):
            color = (0, 255, 0) if label == 1 else (0, 0, 255)
            cv2.circle(disp_img, tuple(point), 6, color, -1)
        
        cv2.imshow('SAM3 Interactive', disp_img)
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('s') and len(points) > 0:
            # 执行分割
            results = sam_model(
                image,
                points=np.array(points),
                labels=np.array(labels)
            )
            
            # 如果有分割结果，显示掩码
            if results[0].masks is not None:
                masks = results[0].masks.data.cpu().numpy()
                for i, mask in enumerate(masks[:3]):  # 显示前3个掩码
                    mask_resized = cv2.resize(mask, (image.shape[1], image.shape[0]))
                    colored_mask = np.zeros_like(image)
                    colored_mask[mask_resized > 0.5] = [0, 255, 0]
                    image = cv2.addWeighted(image, 0.7, colored_mask, 0.3, 0)
    
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()